# Part 2: Data Exploration & Entity Resolution

In this notebook, I showcase the raw data and how I solve the entity resolution problem (linking a raw ad impression to its physical `.png` file).

### 1. The Raw Event Log
First, let's look at the raw impressions and clicks generated by users in the wild.

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

# Load a sample of the 100MB+ inventory file
df_inv = pd.read_csv('../data/campaigns_inventory_updated.csv', nrows=1000)
df_inv[['game_key', 'type', 'device_type', 'platform_os', 'geo_country']].head()

,game_key,type,device_type,platform_os,geo_country
0,adunit-facebook-conversational-commerce-phase-2-thailand-coffeehouse-mob-v2/48e47efe77cf8ba829f3,impression,Mobile,6,Thailand
1,adunit-facebook-conversational-commerce-phase-2-thailand-coffeehouse-mob-v2/48e47efe77cf8ba829f3,impression,Mobile,6,Thailand
2,adunit-facebook-conversational-commerce-phase-2-thailand-baanraya-mob-v2/a8656a5a1477a8cac5f0,impression,Mobile,6,Thailand
3,adunit-facebook-conversational-commerce-phase-2-thailand-coffeehouse-mob-v2/48e47efe77cf8ba829f3,impression,Mobile,6,Thailand
4,adunit-facebook-conversational-commerce-phase-2-thailand-coffeehouse-mob-v2/48e47efe77cf8ba829f3,impression,Mobile,6,Thailand


### 2. The Creative Assets
Now, let's look at the physical images provided by the creative team. Notice how the filenames are completely different from the `game_key` in the log above!

- Inventory game_key format: `<slug> / <request_id>`
- Create asset filename: `<slug> - <request_id>.png`

In [2]:
import os

images = os.listdir('../data/Creative Assets_')
print(f"Total Images: {len(images)}\n")

print("Sample Filenames: [")
for image in images[:5]:
    print(f"    '{image}',")
print("]")

Total Images: 144

Sample Filenames: [
    'adunit-iwc-portugieser-physics-mob-d071433c0e09216d8f80.png',
    'adunit-iwc-portugieser-physics-video-mob-64aa8a100279726c7a76.png',
    'adunit-iwc-portugieser-physics-video-mpu-9159215be71a7409cc23.png',
    'adunit-lionsgate-spiral-new-2-puzzle-v3-mob-512400c815bdcaa21b4f.png',
    'adunit-lionsgate-spiral-new-2-puzzle-v5-mpu-fb0ea07e00d0e2f707e3.png',
]


### 3. The Entity Resolution Bridge
The `request_id` (the 20-character hex code) is our 'Rosetta Stone'. The inventory CSV has it (hidden after the `/`), the image filenames have it (hidden after the `-`), and the global design JSON uses it as a key. By extracting the `request_id` from all three datasets, we can successfully join them together!

In [3]:
import json

with open('../data/global_design_data.json', 'r') as f:
    design_data = json.load(f)

# Show a sample from the JSON metadata
sample_md5 = list(design_data.keys())[0]
sample_req_id = list(design_data[sample_md5].keys())[0]
print(f"JSON MD5 Key: {sample_md5}")
print(f"Contains Metadata for Request ID: {sample_req_id}\n")

# Display the actual metadata content (colors, labels, video info, etc.)
print(json.dumps(design_data[sample_md5][sample_req_id], indent=2))

JSON MD5 Key: f910338e60315f02812b05e932068f47
Contains Metadata for Request ID: 1028d033d74dd2e24f67

{
  "labels": {
    "engagement": [
      "Asphalt",
      "Tarmac",
      "Road",
      "Runway"
    ],
    "click_through": [
      "Asphalt",
      "Tarmac",
      "Road",
      "Runway"
    ]
  },
  "text": {
    "engagement": [
      "Amazing Hong Kong",
      "air",
      "deals!",
      "Swipe Up",
      "BEST",
      "OF ALL",
      "IT'S IN",
      "HONG",
      "KONG"
    ],
    "click_through": [
      "Amazing Hong Kong",
      "air",
      "deals!",
      "Swipe Up",
      "BEST",
      "OF ALL",
      "IT'S IN",
      "HONG",
      "KONG"
    ]
  },
  "colors": {
    "engagement": {
      "1": {
        "red": 153,
        "green": 154,
        "blue": 167,
        "proportion": 0.21955128205128205,
        "saturation": 0.07368421052631581,
        "luminosity": 0.6274509803921569
      },
      "2": {
        "red": 32,
        "green": 43,
        "blue": 66,
        

### 4. Computing the Target Metrics
By successfully linking these datasets, we can aggregate raw impressions and clicks into our target metrics: **Engagement Rate (ER)** and **Click-Through Rate (CTR)**.

In [8]:
import sys
sys.path.append("..")

from src.ingestion.entity_resolution import build_creative_kpi_table, build_linked_dataset
from src.ingestion.loader import load_inventory, load_global_design, list_creative_images

# This is just an illustrative cell for the demo to show the functions exist.
print("Run the full pipeline using `python -m src.pipeline.run_all` to see this in action at scale!")

Run the full pipeline using `python -m src.pipeline.run_all` to see this in action at scale!
